In [ ]:
# Required Libraries
import json
import hashlib
from datetime import datetime
from pathlib import Path
from typing import Optional, Dict, Any, List

# ChromaDB for vector storage
try:
    import chromadb
    CHROMA_AVAILABLE = True
except ImportError:
    CHROMA_AVAILABLE = False
    print("⚠️ ChromaDB not installed. Run: pip install chromadb")

## 1. Simple JSON-based Memory (Fallback)

In [ ]:
class SimpleMemory:
    """
    A simple JSON-based memory system.
    Stores data locally in a JSON file.
    """
    
    def __init__(self, storage_path: str = "./memory_store.json"):
        self.storage_path = Path(storage_path)
        self.data: Dict[str, List[Dict]] = {}
        self._load()
    
    def _load(self):
        """Load existing data from file."""
        if self.storage_path.exists():
            with open(self.storage_path, 'r') as f:
                self.data = json.load(f)
    
    def _save(self):
        """Save data to file."""
        self.storage_path.parent.mkdir(parents=True, exist_ok=True)
        with open(self.storage_path, 'w') as f:
            json.dump(self.data, f, indent=2, default=str)
    
    def store(self, category: str, content: str, dataset_id: str, 
              metadata: Optional[Dict] = None) -> str:
        """Store a memory entry."""
        key = f"{category}_{dataset_id}"
        
        if key not in self.data:
            self.data[key] = []
        
        entry = {
            "id": hashlib.md5(f"{content}{datetime.now()}".encode()).hexdigest()[:8],
            "content": content,
            "dataset_id": dataset_id,
            "category": category,
            "timestamp": datetime.now().isoformat(),
            "metadata": metadata or {}
        }
        
        self.data[key].append(entry)
        self._save()
        
        return entry["id"]
    
    def get_latest(self, category: str, dataset_id: str) -> Optional[Dict]:
        """Get the most recent entry for a category and dataset."""
        key = f"{category}_{dataset_id}"
        
        if key in self.data and self.data[key]:
            return self.data[key][-1]
        return None
    
    def get_all(self, category: str, dataset_id: str) -> List[Dict]:
        """Get all entries for a category and dataset."""
        key = f"{category}_{dataset_id}"
        return self.data.get(key, [])
    
    def search(self, query: str, category: Optional[str] = None) -> List[Dict]:
        """Simple keyword search across all entries."""
        results = []
        query_lower = query.lower()
        
        for key, entries in self.data.items():
            if category and not key.startswith(category):
                continue
            
            for entry in entries:
                if query_lower in entry.get("content", "").lower():
                    results.append(entry)
        
        return results

## 2. ChromaDB-based Memory (Vector Search)

In [ ]:
class VectorMemory:
    """
    ChromaDB-based memory with semantic search.
    Uses embeddings for similarity-based retrieval.
    """
    
    def __init__(self, persist_directory: str = "./vectorstore"):
        if not CHROMA_AVAILABLE:
            raise ImportError("ChromaDB not available")
        
        self.persist_dir = Path(persist_directory)
        self.persist_dir.mkdir(parents=True, exist_ok=True)
        
        # Initialize ChromaDB client
        self.client = chromadb.PersistentClient(path=str(self.persist_dir))
        
        # Get or create collection
        self.collection = self.client.get_or_create_collection(
            name="agent_memory",
            metadata={"description": "Agent conversation memory"}
        )
        
        print(f"✅ VectorMemory initialized at {self.persist_dir}")
    
    def store(self, category: str, content: str, dataset_id: str,
              metadata: Optional[Dict] = None) -> str:
        """Store content with embeddings."""
        doc_id = hashlib.md5(f"{content}{datetime.now()}".encode()).hexdigest()[:12]
        
        full_metadata = {
            "category": category,
            "dataset_id": dataset_id,
            "timestamp": datetime.now().isoformat(),
            **(metadata or {})
        }
        
        self.collection.add(
            documents=[content],
            metadatas=[full_metadata],
            ids=[doc_id]
        )
        
        return doc_id
    
    def search(self, query: str, n_results: int = 5, 
               category: Optional[str] = None,
               dataset_id: Optional[str] = None) -> List[Dict]:
        """Semantic search for similar content."""
        where_filter = {}
        if category:
            where_filter["category"] = category
        if dataset_id:
            where_filter["dataset_id"] = dataset_id
        
        results = self.collection.query(
            query_texts=[query],
            n_results=n_results,
            where=where_filter if where_filter else None
        )
        
        # Format results
        formatted = []
        if results["documents"]:
            for i, doc in enumerate(results["documents"][0]):
                formatted.append({
                    "content": doc,
                    "metadata": results["metadatas"][0][i] if results["metadatas"] else {},
                    "id": results["ids"][0][i] if results["ids"] else None,
                    "distance": results["distances"][0][i] if results.get("distances") else None
                })
        
        return formatted
    
    def get_latest(self, category: str, dataset_id: str) -> Optional[Dict]:
        """Get the most recent entry for a category."""
        results = self.collection.get(
            where={"$and": [
                {"category": category},
                {"dataset_id": dataset_id}
            ]}
        )
        
        if results["documents"]:
            # Sort by timestamp and get latest
            entries = list(zip(results["documents"], results["metadatas"], results["ids"]))
            entries.sort(key=lambda x: x[1].get("timestamp", ""), reverse=True)
            
            return {
                "content": entries[0][0],
                "metadata": entries[0][1],
                "id": entries[0][2]
            }
        return None

## 3. Test Simple Memory

In [ ]:
# Create simple memory
simple_mem = SimpleMemory("./demo_memory.json")

# Store some data
id1 = simple_mem.store(
    category="eda_summary",
    content="Dataset has 891 rows, 12 columns. Target: Survived (binary)",
    dataset_id="titanic",
    metadata={"rows": 891, "columns": 12}
)
print(f"✅ Stored entry: {id1}")

id2 = simple_mem.store(
    category="model_results",
    content="Random Forest achieved 85% accuracy with ROC-AUC of 0.89",
    dataset_id="titanic",
    metadata={"accuracy": 0.85, "roc_auc": 0.89}
)
print(f"✅ Stored entry: {id2}")

In [ ]:
# Retrieve latest
latest = simple_mem.get_latest("eda_summary", "titanic")
print("\n📥 Latest EDA Summary:")
print(f"  Content: {latest['content']}")
print(f"  Metadata: {latest['metadata']}")

In [ ]:
# Search
results = simple_mem.search("accuracy")
print("\n🔍 Search results for 'accuracy':")
for r in results:
    print(f"  - {r['content'][:60]}...")

## 4. Test Vector Memory (ChromaDB)

In [ ]:
if CHROMA_AVAILABLE:
    # Create vector memory
    vector_mem = VectorMemory("./demo_vectorstore")
    
    # Store some data
    vector_mem.store(
        category="conversation",
        content="The data scientist recommended using Random Forest for classification",
        dataset_id="titanic"
    )
    
    vector_mem.store(
        category="conversation",
        content="Hyperparameter tuning improved the model accuracy by 5%",
        dataset_id="titanic"
    )
    
    vector_mem.store(
        category="conversation",
        content="Feature engineering: Created FamilySize and Title features",
        dataset_id="titanic"
    )
    
    print("✅ Stored 3 entries in vector memory")
else:
    print("⚠️ Skipping vector memory test - ChromaDB not available")

In [ ]:
if CHROMA_AVAILABLE:
    # Semantic search
    print("\n🔍 Semantic search for 'model performance':")
    results = vector_mem.search("model performance", n_results=3)
    for i, r in enumerate(results, 1):
        print(f"  {i}. {r['content']}")
        if r.get('distance'):
            print(f"     (distance: {r['distance']:.3f})")

In [ ]:
if CHROMA_AVAILABLE:
    # Search for feature engineering
    print("\n🔍 Semantic search for 'new features created':")
    results = vector_mem.search("new features created", n_results=2)
    for i, r in enumerate(results, 1):
        print(f"  {i}. {r['content']}")

## 5. Cleanup Demo Files

In [ ]:
import shutil

# Clean up demo files
demo_files = ["./demo_memory.json"]
demo_dirs = ["./demo_vectorstore"]

for f in demo_files:
    if Path(f).exists():
        Path(f).unlink()
        print(f"🗑️ Removed {f}")

for d in demo_dirs:
    if Path(d).exists():
        shutil.rmtree(d)
        print(f"🗑️ Removed {d}")

print("\n✅ Cleanup complete!")

## ✅ Summary

This module provides two memory systems:

**1. SimpleMemory (JSON-based)**
- `store()` - Save content with metadata
- `get_latest()` - Get most recent entry
- `get_all()` - Get all entries
- `search()` - Keyword search

**2. VectorMemory (ChromaDB)**
- `store()` - Save with embeddings
- `search()` - Semantic similarity search
- `get_latest()` - Get most recent entry

Use SimpleMemory for basic needs, VectorMemory for semantic search.